### Simulating Hamiltonian PDEs

In [1]:
!pip install imageio[ffmpeg]
!pip install imageio[pyav]
from IPython.display import display, Video, Image
from typing import Callable
from scipy.sparse import diags_array
from tqdm import tqdm
from scipy.integrate import solve_ivp
import imageio
import matplotlib.pyplot as plt
import numpy as np
import torch

ttype = torch.float32
torch.set_default_dtype(ttype)

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [2]:
def create_video(arrays, labels, x_axis=None, file_name='animation.mp4', fps=100, dpi=100, output_format='MP4'):
    """
    Creates an MP4 video or GIF showing the evaluation of a system over time, given by data in an
    array where the time scale is along the first dimension.

    Parameters
    ----------
    arrays : list of ndarray
        List of numpy arrays containing the data to be plotted.
    labels : list of str
        List of labels for the data series, used in the legend of the plot, corresponding to the
        list of arrays.
    x_axis : ndarray, optional
        x-axis values. If provided, data will be plotted against these x-axis values. If None,
        data will be plotted against the array indices.
    file_name : str, optional
        Name of the output animation file. Defaults to 'animation.mp4'.
    fps : int, optional
        Frames per second for the animation. Defaults to 10.
    dpi : int, optional
        Dots per inch for the plot's resolution. Defaults to 100.
    output_format : str, optional
        Output format for the animation. Can be 'MP4' or 'GIF'.
        Defaults to 'MP4'.

    Returns
    -------
    Video or None
        Returns a Video object if the output_format is 'MP4'. If output_format is 'GIF',
        the GIF is displayed and None is returned.
    """

    fig, ax = plt.subplots(figsize=(7.04, 4), dpi=dpi)
    colors = [(0,0,0),(0,0.4,1),(1,0.7,0.3),(0.2,0.7,0.2),(0.8,0,0.2),(0.5,0.3,.9)][:len(arrays)]
    lower_value = min(min(np.min(data) for data in arrays), -2)
    upper_value = max(max(np.max(data) for data in arrays), 2*np.max(arrays[0]))
    limit_value = max(np.abs(lower_value), np.abs(upper_value))
    ax.set_ylim(-limit_value, limit_value)

    with imageio.get_writer(file_name, mode='I', fps=fps) as writer:
        for frame in range(arrays[0].shape[0]):
            lines = []
            for (data, color, label) in zip(arrays, colors, labels):
                if x_axis is not None:
                    line, = ax.plot(x_axis, data[frame, :], color=color, label=label)
                else:
                    line, = ax.plot(data[frame, :], color=color, label=label)
                lines.append(line)
            ax.legend(loc='upper right')
            ax.set_xlabel('$x$', fontsize=12)
            ax.set_ylabel('$u$', fontsize=12)
            fig.canvas.draw()
            image = np.frombuffer(fig.canvas.tostring_rgb(), dtype='uint8')
            image = image.reshape(fig.canvas.get_width_height()[::-1] + (3,))
            writer.append_data(image)
            ax.clear()
            ax.set_ylim(-limit_value, limit_value)
        plt.close(fig)
    
    if output_format == 'MP4':
        return Video(file_name)
    elif output_format == 'GIF':
        with open(file_name,'rb') as f:
            display(Image(data=f.read(), format='gif'))
        return None

## Korteweg-De Vries equation

The Korteweg-De Vries (KdV) equation is a PDE which serves as a mathematical model of waves on shallow water surfaces. It models one dimensional nonlinear dispersive nondissipative waves described by a function $u(t,x)$ that satisfies the equation

$$u_t+\eta uu_x-\gamma^2u_{xxx}=0 $$

We let $\eta=6$ and $\gamma=1$ and assume periodic solutions $u(0,t)=u(P,t)$ on the domain $[0,P]$, with $P=20$. 

In [ ]:
# Recommended wave-specific parameters.
eta = 6.0
gamma = 1.0

# Parameters for the spatial dimension
num_spatial_points = 100
period = 20

# Temporal parameters
timestep = 0.0025
max_time = 1
num_trajectories = 1

x = np.linspace(0, period - period / num_spatial_points, num_spatial_points)
t_axis = np.arange(0, max_time, timestep)
M = x.size
dx = (x[-1] - x[0]) / (M - 1)
print(f'The space grid is of width {dx}')
print(f'The temporal grid is of width {timestep}')

We generate training data from initial conditions

$$u(x,0)=\sum_{l=1}^2\left( \frac{12}{\eta}c_l^2 \right)sech^2\left(c_l\left(\left( x+\frac{P}{2}-d_lP \right)-\frac{P}{2}\right)\right) $$

where $c_1,c_2$ and $d_1,d_2$ are randomly drawn from the uniform distributions $\mathcal{U}(0.5,2)$ and $\mathcal{U}(0,1)$ respectively. The initial state are two waves of height $2c_1^2$ and $2c_2^2$ centred at $d_1P$ and $d_2P$, with periodicity imposed.

In [ ]:
# Define the initial condition sampler.
n_solitons = 2

def initial_condition() -> Callable[[np.random.Generator], np.ndarray]:
    P = (x[-1] - x[0]) * M / (M - 1)
    sech = lambda a: 1 / np.cosh(a)

    def sampler_solitons(rng: np.random.Generator) -> np.ndarray:
        ks = rng.uniform(0.5, 2.0, n_solitons)
        ds = rng.uniform(0.0, 1.0, n_solitons)
        u0 = np.zeros_like(x)

        for k, d in zip(ks, ds):
            center = (x + P / 2 - d * P) % P - P / 2
            coeff = 12.0 / eta * k**2
            u0 += coeff * sech(np.abs(k * center)) ** 2

        #u0 = np.concatenate([u0[M:], u0[:M]], axis=-1)
        return u0
    
    return sampler_solitons

The KdV equation admits the following Hamiltonian functional

$$ \mathcal{H}[u]=-\int_{\Omega}\left(\frac{\eta}{6}u^3+\frac{\gamma^2}{2}u_x^2 \right)dx $$

During simulation and after defining our spatial discretization, this integral is approximated by 

$$ \mathcal{H}[u]\approx -h\sum_{i=0}^{M-1}\left(\frac{\eta}{6}u_i^3+\frac{\gamma^2}{2}(\delta_fu_i)² \right) $$

where the operator $\delta_f$ denotes forward difference, i.e. $\delta_fu_i=(u_{i+1}-u_{i-1})/(2h)$ 

$$S_d=\frac{1}{2h}\begin{bmatrix} 0 & 1 & 0 & \cdots & 0 & -1 \\ -1 & 0 & 1 & 0 & \cdots & \\ 0 & -1 & 0 & 1 & 0 & \cdots \\ & & \ddots & \ddots & \ddots & \\ & \cdots & 0 & -1 & 0 & 1 \\ 1 & 0 & \cdots & 0 & -1 & 0  \end{bmatrix} $$

where the first and last rows are adjusted according to the periodic boundary conditions.

In [ ]:
# 1st order forward difference matrix.
Dp = diags_array([1, -1, 1], offsets=[-M + 1, 0, 1], shape=(M, M)).toarray()
Dp = 1 / dx * Dp
# 1st order central difference matrix.
D1 = diags_array([1, -1, 1, -1], offsets=[-M + 1, -1, 1, M - 1], shape=(M, M)).toarray()
D1 = 0.5 / dx * D1
# 2nd order central difference matrix.
D2 = diags_array([1, 1, -2, 1, 1], offsets=[-M + 1, -1, 0, 1, M - 1], shape=(M, M)).toarray()
D2 = 1 / dx**2 * D2

# Define the Hamiltonian and its gradient.
def hamiltonian(u: np.ndarray) -> np.ndarray:
    integrand = -1 / 6 * eta * u**3 + 0.5 * gamma **2 * ((u @ Dp) ** 2)
    return np.sum(integrand, axis=-1)

def hamiltonian_grad(u: np.ndarray) -> np.ndarray:
    return -0.5 * eta * u**2 - (gamma**2 * u @ D2)

Simulate the trajectory taking into account that we can rewrite the KdV equation as 

$$u_t=S\frac{\partial\mathcal{H}}{\partial u}[u]-R\frac{\partial V}{\partial u}[u]+f(u,x,t) $$

with $S=\frac{\partial}{\partial x}$, $R=0$ and $f(u,x,t)=0$.

In [ ]:
traj_length = t_axis.shape[0]
x_ = np.zeros((num_trajectories, traj_length, num_spatial_points))
dxdt = np.zeros((num_trajectories, traj_length, num_spatial_points))
t = np.zeros((num_trajectories, traj_length))
u = np.zeros((num_trajectories, traj_length - 1, num_spatial_points))

dt = torch.tensor([t[0, 1] - t[0, 0]], dtype=ttype)
for i in tqdm(range(num_trajectories)):
    # Define the initial conditions.
    rng = np.random.default_rng(42)
    x0 = initial_condition()(rng)
    # Define the time derivative of the state.
    S = D1
    x_dot = lambda t, x: hamiltonian_grad(x) @ (S.T)
    # Generate the trajectory using scipy.
    out_ivp = solve_ivp(
        fun=x_dot, t_span=(t_axis[0], t_axis[-1]), y0=x0, t_eval=t_axis, rtol=1e-10
    )
    x_[i], t[i] = out_ivp["y"].T, out_ivp["t"].T

In [ ]:
create_video(
    [x_[0]],
    ["Ground truth"],
    x_axis=x
)

## Conservative string equation

The following notebook shows how to simulate the string equation with fixed edges. The conservative string equation is a particularization of the wave equation, whose behavior is described as 

$$ (\mu\partial_{t}^2-T_0\partial_x^2+EI\partial_{x}^4)u=f(t,x) $$

where $\mu=\rho A$ denotes the lineic density of mass, $T$ the tension and $I$ the inertia momentum. We will impose Dirichlet boundary conditions ($w(t,x)=0$ on $(0,\infty)\times\{0,L\}$) and zero initial conditions ($w=0$ on $\{0\}\times(0,L)$). We consider the spatial domain $[0,0.65]$ and simulate from $t=0$ to $t=2$ with a time step $\Delta t=0.0025$.



In [3]:
# Parameters for the spatial dimension
num_spatial_points = 500
num_pi = num_spatial_points - 2
num_strain = num_spatial_points - 1

# String length in which to make the simulation.
period = 0.65

# Temporal parameters
timestep = 0.0025
max_time = 1
num_trajectories = 1

# Create the space and temporal grid.
x = np.linspace(0, period - period / num_spatial_points, num_spatial_points)
t_axis = np.arange(0, max_time, timestep)
M = x.size
dx = (x[-1] - x[0]) / (M - 1)

# Print basic information about the grid.
print(f'The space grid is of width {dx:.2e}.')
print(f'The time grid is of width {timestep:.2e}.')

The space grid is of width 1.30e-03.
The time grid is of width 2.50e-03.


We will transform this 2nd-order in time PDE into a 1st-order in time PDE. For this purpose, we introduce the states 

$$\alpha:=\begin{bmatrix}\pi:=\mu \partial_tu \\ \epsilon:=\partial_xw \\ \epsilon_x:=\partial_x^2u \end{bmatrix}$$

where $\pi$ denotes the lineic momentum and $\epsilon$ the strain. This allow us to write the Hamiltonian of the system as 

$$H(t)=\int_0^L \left(\frac{1}{2\mu}\partial_t^2u+\frac{T}{2}\partial_x^2u+\frac{EI}{2}\partial_x^4u \right)dx= \int_0^L \left(\frac{\alpha_0^2}{2\mu}+\frac{T}{2}\alpha_1^2+\frac{EI}{2}\alpha_2^2\right) dx$$

This is the continuous definition, which in the simulation is approximated as 

$$H(t)\approx h\sum_{i=0}^{M-1} \left(\frac{\alpha_{0,i}^2}{2}+\frac{T}{2}\alpha_{1,i}^2+ \frac{EI}{2}\alpha_{2,i}^2\right)=\frac{1}{2}\|\alpha_0 \|^2+ \frac{T}{2}\|\alpha_1\|^2+\frac{EI}{2}\|\alpha_2\|^2$$

In [4]:
# Parameters from a guitar string (E4).
rho = 7850
R = 127e-3
A = np.pi * R**2 
mu = rho * A
tension = 73.75
I = np.pi * R**4 / 4
E = 2.1e11

# Define the Hamiltonian and its gradient.
def hamiltonian(x: np.ndarray, mu=mu, T=tension) -> np.ndarray:
    integrand = (mu * x[..., 0]**2) + (T * x[..., 1]**2)
    return 0.5 * np.sum(integrand, axis=-1)

def hamiltonian_grad(x: np.ndarray) -> np.ndarray:
    e_pi = (1 / mu) * x[..., :num_pi]
    e_strain = tension * x[..., num_pi:num_pi + num_strain]
    e_strain_x = E * I * x[..., num_pi + num_strain:]
    return dx * np.concatenate((e_pi, e_strain, e_strain_x), axis=-1)

The port-Hamiltonian formulation of the conservative string equation can then simply be written as 

$$ \begin{bmatrix} \pi_t \\ \epsilon_t   \end{bmatrix} = \begin{bmatrix} 0 & \partial_x \\ \partial_x & 0 \end{bmatrix}\begin{bmatrix} \delta_{\pi}H \\ \delta_{\epsilon}H \end{bmatrix} $$

**Spatial disretization**. As fixed boundary conditions are considered ($u_0(t)=u_N(t)=0,\forall t\in\mathbb{R}^{+}$), the transverse displacement $u(x,t)$ is approximated by the displacements at the $N-1$ interior nodes of the grid

$$\mathbf{u}(t)=[u_1(t),\cdots,u_i(t),\cdots,u_{N-1}(t)]^T $$ 

 In order to build derivatives of the displacement inside the interconnection matrix, a finite difference matrix $D^{-}$ of first oder is used with Dirichlet boundary conditions.

$$ D^{-}=\frac{1}{h}\begin{bmatrix} 1 & 0 & 0  \\  -1 & \ddots & 0  \\ 0 & \ddots & 1  \\  0 & 0 & -1    \end{bmatrix}\in\mathbb{R}^{dim(\epsilon)\times dim(\pi)}$$

where $dim(\epsilon)=N$ and $dim(\pi)=N-1$. This rectangle matrix generates an approximation of the first spatial derivative of the displacement at the nodes of a "dual" grid, which is staggered with respect to the primal grid. As $\mathbf{u}$ is defined on the primal grid, the momentum $\mathbf{p}\equiv \mu u_t$ is also defined on the primal grid. The first spatial derivative of the displacement is obtained as $\epsilon=D^{-}\mathbf{u}$ and is thus defined on the dual grid.

Note: See **Finite differences on staggered grids preserving the port-Hamiltonian structure with application to an acoustic duct** (https://www.sciencedirect.com/science/article/abs/pii/S0021999118304297) for a complete introduction to this dual grid concept.

Finally, the second spatial derivative of the displacement is obtained as $\partial_{xx}=D^{+}\epsilon$ with 

$$ D^{+}=-(D^{-})^T\in\mathbb{R}^{dim(\pi)\times dim(\epsilon)} $$

In [5]:
import einops

# Forward difference operator with Dirichlet conditions.
D_plus =  diags_array([-1, 1], offsets=[0, 1], shape=(num_pi, num_strain)).toarray()
# Define the adjoint operator.
D_minus = - D_plus.T
# Rescale with the space grid width.
D_plus = 1 / dx * D_plus
D_minus = 1 / dx * D_minus
# Define the second derivative.
D2 = einops.einsum(D_plus, D_minus, 'i j, j k -> i k')

The discrete system dynamic then directly follows as

$$ \begin{bmatrix} \pi_t \\ \epsilon_t \\ \epsilon_{x,t} \end{bmatrix}=\begin{bmatrix} \mathbf{0}^{(N-1)\times (N-1)} & D^{+} & -D^2 \\ D^{-} & \mathbf{0}^{N\times N} & \mathbf{0}^{N\times N-1} \\ D^2 & \mathbf{0}^{N-1\times N} & \mathbf{0}^{N-1\times N-1} \end{bmatrix}\begin{bmatrix} \delta_{\pi}H \\ \delta_{\epsilon}H \\ \delta_{\epsilon_x}H \end{bmatrix}$$

The system simulation will be initialize with $\pi(0,x)=0$ and 

$$ \epsilon(0,x)= \begin{cases} \frac{c_0\pi}{2x_{hw}}(-sin(\pi(x-x_0)/x_{hw})) \hspace{0.5cm} \text{if} \hspace{0.2cm} |x-x_0|\leq x_{hw}  \\ 0 \hspace{0.5cm} \text{else}\end{cases}$$

where $c_0$ is the peak amplitude, $x_0$ the spatial center, and $x_{hw}$ the half-width length of the spatial distribution.

In [6]:
# Define the staggered grid (dual grid) for the strains.
xtilde = (x[:-1] + x[1:])/2

# Define the initial condition sampler.
def initial_condition() -> Callable[[np.random.Generator], np.ndarray]:
    P = 1

    def sampler_solitons(rng: np.random.Generator) -> np.ndarray:
        amplitude_peak = rng.uniform(0.5, 2.0, 1)
        peak_location = rng.uniform(0.0, 1.0, 1)
        x_hw = 0.5
        u0 = np.zeros_like(xtilde)

        for c0, x0 in zip(amplitude_peak, peak_location):
            center = - np.sin(np.pi * (xtilde - 0.325) / x_hw)
            coeff = (0.05 * np.pi) / (2 * x_hw)
            u0 += coeff * center

        return u0
    
    return sampler_solitons

Simulate the trajectory using a RK4 solver and the spatial discretization mentioned above.

In [7]:
traj_length = t_axis.shape[0]
energy_states = np.zeros((num_trajectories, traj_length, 2 * num_pi + num_strain))
dxdt = np.zeros((num_trajectories, traj_length, 2 * num_pi + num_strain))
t = np.zeros((num_trajectories, traj_length))

dt = torch.tensor([t[0, 1] - t[0, 0]], dtype=ttype)

for i in tqdm(range(num_trajectories)):
    # Define the initial conditions.
    epsilon = initial_condition()(np.random)
    epsilon_x = einops.einsum(D_plus, epsilon, 'i j, j -> i')
    pi = np.zeros(num_pi)
    x0 = np.concatenate((pi, epsilon, epsilon_x), axis=-1)
    # Define the time derivative of the state.
    S = np.block(
        [
            [np.zeros((num_pi, num_pi)), D_plus, -D2],
            [D_minus, np.zeros((num_strain, num_strain)), np.zeros((num_strain, num_pi))],
            [D2, np.zeros((num_pi, num_strain)), np.zeros((num_pi, num_pi))]
        ]
    )
    x_dot = lambda t, x: hamiltonian_grad(x).flatten() @ (S.T)
    # Generate the trajectory using scipy.
    out_ivp = solve_ivp(
        fun=x_dot, t_span=(t_axis[0], t_axis[-1]), y0=x0, t_eval=t_axis, rtol=1e-10
    )
    energy_states[i], t[i] = out_ivp["y"].T, out_ivp["t"].T

plt.plot(epsilon)

  0%|          | 0/1 [12:02<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
create_video(
    [energy_states[0, ..., :num_pi]],
    ["Lineic momentum"],
    x_axis=x[1:-1],
    fps=int(1/timestep)
)